# Dippy Animation Trajectory Generator

This notebook launches the Dippy app — a Gradio interface for chained WAN 2.1 Image-to-Video generation with frame continuity.

**Requirements:** A Colab runtime with GPU (T4 or better). An HF token and optionally an OpenAI API key configured in Colab Secrets.

In [ ]:
# Cell 1: Mount Google Drive (for model caching)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
# Move out of the directory before deleting it to avoid 'getcwd' errors
os.chdir('/content')
!rm -rf /content/dippy-WAN/

In [ ]:
import os
os.chdir('/content')

In [ ]:
!mkdir /root/.ssh/

# Copy '/content/drive/Colab Notebooks/utils/id_ed25519' to /root/.ssh
!cp '/content/drive/MyDrive/Colab Notebooks/utils/id_ed25519' /root/.ssh/.
!chmod 700 /root/.ssh
!chmod 600 /root/.ssh/id_ed25519
!ssh-keyscan -t ed25519 github.com >> /root/.ssh/known_hosts
!chmod 644 /root/.ssh/known_hosts
!ssh -T git@github.com || true

# github.com:22 SSH-2.0-b1ddf78
Hi dfu99! You've successfully authenticated, but GitHub does not provide shell access.


In [ ]:
import os
os.chdir('/content')

if not os.path.exists('/content/dippy-WAN'):
    # Option A: Clone from GitHub (update URL to your repo)
    !git clone git@github.com:dfu99/dippy-WAN.git /content/dippy-WAN
else:
    print('dippy-WAN directory found.')

Cloning into '/content/dippy-WAN'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 48 (delta 19), reused 43 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 48.20 KiB | 214.00 KiB/s, done.
Resolving deltas: 100% (19/19), done.


In [ ]:
# Cell 3: Install latest-compatible dependencies (WAN + Transformers 5)
!pip install -q --upgrade \
  "diffusers==0.36.0" \
  "transformers==5.1.0" \
  "accelerate==1.12.0" \
  "huggingface_hub==1.4.1" \
  "gradio==6.5.1" \
  spaces ftfy peft imageio-ffmpeg opencv-python safetensors sentencepiece openai

# Print resolved versions so mismatches are obvious
import importlib.metadata as md
for pkg in ("diffusers", "transformers", "accelerate", "huggingface_hub", "gradio"):
    try:
        print(f"{pkg}=={md.version(pkg)}")
    except Exception:
        print(f"{pkg} not installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 144.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.2/105.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.2 MB/s eta 0:00:00
diffusers==0.36.0
transformers==5.1.0
accelerate==1.12.0
huggingface_hub==1.4.1
gradio==6.5.1


In [ ]:
# Cell 4: Configure cache, sync Drive -> local SSD, and set API keys
import os
from google.colab import userdata

DRIVE_CACHE_DIR = '/content/drive/My Drive/huggingface_cache_store'
LOCAL_CACHE_DIR = '/content/hf_cache'
# LOCAL_CACHE_DIR = DRIVE_CACHE_DIR
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)

# print('Syncing existing cache from Drive to local SSD...')
# !rsync -a --ignore-existing --info=stats1 "{DRIVE_CACHE_DIR}/" "{LOCAL_CACHE_DIR}/"

# print('Removing tiny/corrupt safetensors placeholders from local cache...')
# !find "{LOCAL_CACHE_DIR}" -type f -name "*.safetensors" -size -16k -print -delete

# Use local SSD for runtime I/O reliability and speed
os.environ['HF_HOME'] = LOCAL_CACHE_DIR
os.environ['HF_HUB_CACHE'] = LOCAL_CACHE_DIR
os.environ['HF_ASSETS_CACHE'] = os.path.join(LOCAL_CACHE_DIR, 'assets')
os.environ['HF_XET_CACHE'] = os.path.join(LOCAL_CACHE_DIR, 'xet')
for key in ('HF_HOME', 'HF_HUB_CACHE', 'HF_ASSETS_CACHE', 'HF_XET_CACHE'):
    os.makedirs(os.environ[key], exist_ok=True)

# Keep Drive path so we can sync back later
os.environ['DIPPY_DRIVE_CACHE_DIR'] = DRIVE_CACHE_DIR

print(f"HF_HOME: {os.environ['HF_HOME']}")
print(f"HF_HUB_CACHE: {os.environ['HF_HUB_CACHE']}")
print(f"HF_ASSETS_CACHE: {os.environ['HF_ASSETS_CACHE']}")
print(f"HF_XET_CACHE: {os.environ['HF_XET_CACHE']}")
print(f"DIPPY_DRIVE_CACHE_DIR: {os.environ['DIPPY_DRIVE_CACHE_DIR']}")

# Hugging Face token (required for model download)
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded successfully.')
except Exception as e:
    print(f'Could not load HF_TOKEN: {e}')
    print('Please add a secret named HF_TOKEN in the Colab secrets tab (key icon).')

# OpenAI API key (optional, for LLM sentence generation)
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('OPENAI_API_KEY loaded successfully.')
except Exception as e:
    print(f'Could not load OPENAI_API_KEY: {e}')
    print('Sentence generation via LLM will not be available.')
    print('You can still type sentences manually.')


HF_HOME: /content/hf_cache
HF_HUB_CACHE: /content/hf_cache
HF_ASSETS_CACHE: /content/hf_cache/assets
HF_XET_CACHE: /content/hf_cache/xet
DIPPY_DRIVE_CACHE_DIR: /content/drive/My Drive/huggingface_cache_store
HF_TOKEN loaded successfully.
OPENAI_API_KEY loaded successfully.


In [ ]:
# Cell 5: Launch the Dippy app (inline in Colab)
import os
import runpy

os.chdir('/content/dippy-WAN')
runpy.run_path('/content/dippy-WAN/dippy-app.py', run_name='__main__')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Cache directories:
- HF_HOME: /content/hf_cache
- HF_HUB_CACHE: /content/hf_cache
- HF_ASSETS_CACHE: /content/hf_cache/assets
- HF_XET_CACHE: /content/hf_cache/xet
Runtime package versions:
- diffusers: 0.36.0
- transformers: 5.1.0
- accelerate: 1.12.0
- huggingface_hub: 1.4.1
- gradio: 6.5.1
Recommended Colab install cell:
!pip install -q --upgrade diffusers==0.36.0 transformers==5.1.0 accelerate==1.12.0 huggingface_hub==1.4.1 gradio==6.5.1 safetensors sentencepiece peft ftfy imageio-ffmpeg opencv-python


config.json:   0%|          | 0.00/582 [00:00<?, ?B/s]

image_encoder/model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

...

Text encoder embeddings are not tied. Attempting repair...
Applied embedding repair by copying shared -> encoder.embed_tokens. Continuing startup.


Wan21_CausVid_14B_T2V_lora_rank32.safete(…):   0%|          | 0.00/319M [00:00<?, ?B/s]

Detected Colab runtime: enabling inline app rendering.
Installed gradio tunnel binary: /usr/local/lib/python3.12/dist-packages/gradio/frpc_linux_amd64_v0.3
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://40665f56f855ac3d8f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  0%|          | 0/4 [00:00<?, ?steps/s]

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2152, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1629, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

In [ ]:
# Cell 6: Sync local SSD cache back to Drive (run after successful model load)
import os

drive_cache = os.environ.get('DIPPY_DRIVE_CACHE_DIR', '/content/drive/My Drive/huggingface_cache_store')
local_cache = os.environ.get('HF_HUB_CACHE', '/content/hf_cache')
os.makedirs(drive_cache, exist_ok=True)

print(f'Syncing local cache to Drive: {local_cache} -> {drive_cache}')
print('Using rsync -L to copy real file contents (avoids broken symlink stubs).')
!rsync -aL --info=stats1 "{local_cache}/" "{drive_cache}/"
print('Drive sync complete.')
